<a href="https://colab.research.google.com/github/YasserJEP/TDA-BifurcacionesHopf/blob/main/Notebooks/4_Exponentes_Lyapunov.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hopf Normal

Este notebook calcula el exponente de Lyapunov máximo (λ₁) para el sistema Hopf normal con μ ∈ [-1,1] (300 valores). Usa integración numérica con ecuación variacional. λ₁ > 0 indica caos, λ₁ = 0 bifurcación, λ₁ < 0 estabilidad. Exporta gráfico PDF y resultados a Excel.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import os

# ======================================================
# 1. PARÁMETROS DEL SISTEMA DE HOPF NORMAL
# ======================================================

omega = 6.0                       # Frecuencia angular
mu_values = np.round(np.linspace(-1, 1, 300), 5)  # Valores del parámetro de bifurcación

# Parámetros de integración
T_trans = 20.0    # Tiempo de transitorio
T_step = 1.0      # Tiempo entre renormalizaciones
N_steps = 1200    # Número de pasos para calcular el exponente
dt = 0.01         # Paso de tiempo

# ======================================================
# 2. SISTEMA DE HOPF NORMAL + ECUACIÓN VARIACIONAL (λ₁)
# ======================================================

def hopf_variational_1(t, Y, mu, omega):
    """
    Sistema aumentado: Hopf normal + ecuación variacional.
    Y = [x, y, vx, vy]
    Retorna las derivadas [dx/dt, dy/dt, dvx/dt, dvy/dt]
    """
    x, y = Y[0:2]      # Estado del sistema original
    v = Y[2:4]         # Vector de perturbación

    r2 = x**2 + y**2   # Radio al cuadrado

    # Ecuaciones del sistema Hopf normal
    fx = mu * x - omega * y - r2 * x
    fy = omega * x + mu * y - r2 * y

    # Matriz Jacobiana del sistema
    J = np.array([
        [mu - 3*x**2 - y**2,   -omega - 2*x*y],
        [omega - 2*x*y,         mu - x**2 - 3*y**2]
    ])

    # Ecuación variacional: dv/dt = J · v
    dv = J @ v

    return [fx, fy, dv[0], dv[1]]

# ======================================================
# 3. CÁLCULO DEL EXPONENTE DE LYAPUNOV MÁXIMO (λ₁)
# ======================================================

lambda_1_list = []

print("\n μ\t\tλ₁")
print("-" * 30)

for mu in mu_values:
    # Condiciones iniciales: x=0, y=0.5
    x0 = np.array([0, 0.5])

    # Vector de perturbación aleatorio normalizado
    v = np.random.rand(2)
    v /= np.linalg.norm(v)

    # Estado aumentado inicial
    Y0 = np.concatenate([x0, v])

    # --- Fase transitoria (eliminar dependencia de condiciones iniciales) ---
    sol_trans = solve_ivp(
        lambda t, y: hopf_variational_1(t, y, mu, omega),
        [0, T_trans],
        Y0,
        max_step=dt
    )

    state = sol_trans.y[:, -1]  # Estado final del transitorio

    sum_log = 0.0  # Acumulador de logaritmos

    # --- Pasos principales para calcular λ₁ ---
    for _ in range(N_steps):
        # Integrar durante T_step
        sol = solve_ivp(
            lambda t, y: hopf_variational_1(t, y, mu, omega),
            [0, T_step],
            state,
            max_step=dt
        )

        state = sol.y[:, -1]  # Estado final

        v = state[2:4]  # Extraer vector de perturbación
        d = np.linalg.norm(v)  # Norma del vector

        v /= d  # Renormalizar
        sum_log += np.log(d)  # Acumular logaritmo de la expansión

        state[2:4] = v  # Actualizar estado con vector renormalizado

    # Calcular exponente de Lyapunov máximo
    lambda_1 = sum_log / (N_steps * T_step)
    lambda_1_list.append(lambda_1)

    print(f"{mu:.5f}\t{lambda_1:.6f}", flush=True)

# ======================================================
# 4. GUARDAR RESULTADOS EN EXCEL
# ======================================================

carpeta = r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\3_EXPONENTES DE LYAPUNOV\HOPF NORMAL"
os.makedirs(carpeta, exist_ok=True)

archivo_excel = os.path.join(
    carpeta,
    f"lyapunov_lambda1_Hopf_Normal_{len(mu_values)}_omega={omega}.xlsx"
)

df = pd.DataFrame({
    "mu": mu_values,
    "lambda_1": lambda_1_list
})
df.to_excel(archivo_excel, index=False)

print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ======================================================
# 5. GRÁFICA DEL EXPONENTE DE LYAPUNOV
# ======================================================

plt.figure(figsize=(10, 3))
plt.plot(mu_values, lambda_1_list, lw=1.2, color="g", label=r"$\lambda_1$")

# Configurar ejes
plt.xlim(-1, 1)
plt.xticks(np.arange(-1, 1.1, 0.25))
plt.margins(x=0.01)

# Etiquetas
plt.xlabel(r"$\mu$", fontsize=13)
plt.ylabel(r"$\lambda_1$", fontsize=13)

# Línea vertical en μ = 0 (punto de bifurcación)
plt.axvline(0.0, linestyle='--', lw=1, alpha=0.9, c="black", label=r"$\mu_c = 0$")

plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()

# Guardar gráfica como PDF
plt.savefig(f"figure_hopf_lyapunov_lambda1_{len(mu_values)}_omega{omega}.pdf",
            bbox_inches="tight", dpi=600)
plt.show()

# Lorenz

Este notebook calcula el exponente de Lyapunov máximo (λ₁) para el sistema Lorenz con ρ ∈ [20,30] (300 valores). Usa integración numérica con ecuación variacional. λ₁ > 0 indica caos, λ₁ ≈ 0 bifurcación. ρ_c ≈ 24.74 marca la transición. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import os

# ======================================================
# 1. PARÁMETROS DEL SISTEMA DE LORENZ
# ======================================================

sigma = 10.0                    # Parámetro sigma
beta = 8.0 / 3.0                # Parámetro beta
rho_values = np.round(np.linspace(20, 30, 300), 6)  # Valores del parámetro rho

# Parámetros de integración para el exponente de Lyapunov
T_trans = 20.0    # Tiempo de transitorio (eliminar dependencia de CI)
T_step = 1.2      # Tiempo entre renormalizaciones
N_steps = 1300    # Número de pasos para calcular el exponente
dt = 0.01         # Paso de tiempo interno del integrador

# ======================================================
# 2. SISTEMA DE LORENZ + ECUACIÓN VARIACIONAL
# ======================================================

def lorenz_variational(t, Y, sigma, beta, rho):
    """
    Sistema aumentado: Lorenz + ecuación variacional.
    Y = [x, y, z, vx, vy, vz]
    Retorna las derivadas [dx/dt, dy/dt, dz/dt, dvx/dt, dvy/dt, dvz/dt]
    """
    x, y, z = Y[0:3]        # Estado del sistema original
    vx, vy, vz = Y[3:6]     # Vector de perturbación

    # Ecuaciones del sistema Lorenz
    fx = sigma * (y - x)
    fy = x * (rho - z) - y
    fz = x * y - beta * z

    # Matriz Jacobiana del sistema
    J = np.array([
        [-sigma,   sigma,    0],
        [rho - z,   -1,     -x],
        [y,         x,    -beta]
    ])

    # Ecuación variacional: dv/dt = J · v
    d_var = J @ np.array([vx, vy, vz])

    return [fx, fy, fz, d_var[0], d_var[1], d_var[2]]

# ======================================================
# 3. CÁLCULO DEL EXPONENTE DE LYAPUNOV MÁXIMO (λ₁)
# ======================================================

lyapunov_max = []

print("\n ρ\t\tλ₁")
print("-" * 30)

for rho in rho_values:
    # Condición inicial en el atractor de Lorenz
    x0 = np.array([6.956, 6.956, 22.82])

    # Estado aumentado inicial con vector de perturbación aleatorio
    Y0 = np.concatenate([x0, np.random.rand(3)])

    # --- Fase transitoria (eliminar dependencia de condiciones iniciales) ---
    sol_trans = solve_ivp(
        lambda t, y: lorenz_variational(t, y, sigma, beta, rho),
        [0, T_trans],
        Y0,
        max_step=dt
    )

    state = sol_trans.y[:, -1]  # Estado final del transitorio

    # Vector de perturbación normalizado
    v = state[3:6]
    v = v / np.linalg.norm(v)
    state[3:6] = v

    lyap_sum = 0.0  # Acumulador de logaritmos

    # --- Pasos principales para calcular λ₁ ---
    for _ in range(N_steps):
        # Integrar durante T_step
        sol = solve_ivp(
            lambda t, y: lorenz_variational(t, y, sigma, beta, rho),
            [0, T_step],
            state,
            max_step=dt
        )

        state = sol.y[:, -1]  # Estado final

        v = state[3:6]        # Extraer vector de perturbación
        d = np.linalg.norm(v)  # Norma del vector

        lyap_sum += np.log(d)  # Acumular logaritmo de la expansión

        # Renormalizar el vector de perturbación
        state[3:6] = v / d

    # Calcular exponente de Lyapunov máximo
    lambda_1 = lyap_sum / (N_steps * T_step)
    lyapunov_max.append(lambda_1)

    print(f"{rho:.6f}\t{lambda_1:.8f}", flush=True)

# ======================================================
# 4. GUARDAR RESULTADOS EN EXCEL
# ======================================================

carpeta = r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\3_EXPONENTES DE LYAPUNOV\LORENZ"
os.makedirs(carpeta, exist_ok=True)

archivo_excel = os.path.join(
    carpeta,
    f"lyapunov_lambda1_Lorenz_{len(rho_values)}_sigma={sigma}_beta={beta:.3f}.xlsx"
)

df = pd.DataFrame({
    "rho": rho_values,
    "lambda_1": lyapunov_max
})
df.to_excel(archivo_excel, index=False)

print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ======================================================
# 5. GRÁFICA DEL EXPONENTE DE LYAPUNOV
# ======================================================

plt.figure(figsize=(10, 3))
plt.plot(rho_values, lyapunov_max, lw=1.2, color="g", label=r"$\lambda_1$")

# Configurar ejes
plt.xlim(20, 30)
plt.xticks(np.arange(20, 30.1, 1))
plt.margins(x=0.01)

# Etiquetas
plt.xlabel(r'$\rho$', fontsize=13)
plt.ylabel(r'$\lambda_1$', fontsize=13)

# Línea vertical en ρ ≈ 24.74 (punto de transición al caos)
plt.axvline(24.74, linestyle='--', lw=1, alpha=0.9, c="black", label=r"$\rho_c \approx 24.74$")

plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()

# Guardar gráfica como PDF
plt.savefig(f"figure_lorenz_lyapunov_lambda1_{len(rho_values)}.pdf",
            bbox_inches="tight", dpi=600)
plt.show()

# Reacción BZ

Este notebook calcula el exponente de Lyapunov máximo (λ₁) para el oscilador de Belousov-Zhabotinsky con b ∈ [2,5] (300 valores). Usa integración numérica con ecuación variacional. λ₁ > 0 indica caos, λ₁ = 0 bifurcación. b_c = 3.5 marca la transición. Exporta gráfico PDF y resultados a Excel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import pandas as pd
import os

# ======================================================
# 1. PARÁMETROS DEL SISTEMA DE BELOUSOV-ZHABOTINSKY (BZ)
# ======================================================

a = 10.0                       # Parámetro a del sistema BZ
b_values = np.round(np.linspace(2, 5, 300), 6)  # Valores del parámetro b

# Parámetros de integración para el exponente de Lyapunov
T_trans = 20.0    # Tiempo de transitorio (eliminar dependencia de CI)
T_step = 1.0      # Tiempo entre renormalizaciones
N_steps = 1300    # Número de pasos para calcular el exponente
dt = 0.01         # Paso de tiempo interno del integrador

# ======================================================
# 2. SISTEMA BZ + ECUACIÓN VARIACIONAL
# ======================================================

def bz_variational(t, Y, a, b):
    """
    Sistema aumentado: BZ + ecuación variacional.
    Y = [x, y, vx, vy]
    Retorna las derivadas [dx/dt, dy/dt, dvx/dt, dvy/dt]
    """
    x, y = Y[0:2]        # Estado del sistema original
    vx, vy = Y[2:4]      # Vector de perturbación

    # ---- Ecuaciones del sistema BZ ----
    dx = a - x - (4 * x * y) / (1 + x**2)
    dy = b * x * (1 - y / (1 + x**2))

    # ---- Matriz Jacobiana del sistema ----
    # Derivadas parciales ∂/∂x y ∂/∂y
    J11 = -1 - (4*y*(1 + x**2) - 8*x**2*y) / (1 + x**2)**2
    J12 = - (4*x) / (1 + x**2)

    J21 = b * (1 - y/(1+x**2)) + b*x * ((2*x*y)/(1+x**2)**2)
    J22 = - b*x / (1 + x**2)

    J = np.array([[J11, J12], [J21, J22]])

    # ---- Ecuación variacional: dv/dt = J · v ----
    dvx = J[0, 0] * vx + J[0, 1] * vy
    dvy = J[1, 0] * vx + J[1, 1] * vy

    return [dx, dy, dvx, dvy]

# ======================================================
# 3. CÁLCULO DEL EXPONENTE DE LYAPUNOV MÁXIMO (λ₁)
# ======================================================

lambda_1_list = []

print("\n b\t\tλ₁")
print("-" * 30)

for b in b_values:
    # Condición inicial del sistema BZ
    x0 = np.array([1.0, 5.1])

    # Vector de perturbación aleatorio normalizado
    v = np.random.rand(2)
    v /= np.linalg.norm(v)

    # Estado aumentado inicial
    Y0 = np.concatenate([x0, v])

    # --- Fase transitoria (eliminar dependencia de condiciones iniciales) ---
    sol_trans = solve_ivp(
        lambda t, y: bz_variational(t, y, a, b),
        [0, T_trans],
        Y0,
        max_step=dt
    )

    state = sol_trans.y[:, -1]  # Estado final del transitorio

    # Vector de perturbación normalizado
    v = state[2:4]
    v /= np.linalg.norm(v)
    state[2:4] = v

    sum_log = 0.0  # Acumulador de logaritmos

    # --- Pasos principales para calcular λ₁ ---
    for _ in range(N_steps):
        # Integrar durante T_step
        sol = solve_ivp(
            lambda t, y: bz_variational(t, y, a, b),
            [0, T_step],
            state,
            max_step=dt
        )

        state = sol.y[:, -1]  # Estado final

        v = state[2:4]        # Extraer vector de perturbación
        d = np.linalg.norm(v)  # Norma del vector

        # Renormalizar y acumular
        v /= d
        sum_log += np.log(d)
        state[2:4] = v

    # Calcular exponente de Lyapunov máximo
    lambda_1 = sum_log / (N_steps * T_step)
    lambda_1_list.append(lambda_1)

    print(f"{b:.5f}\t{lambda_1:.6f}", flush=True)

# ======================================================
# 4. GUARDAR RESULTADOS EN EXCEL
# ======================================================

carpeta = r"D:\MAESTRÍA\CÓDIGOS TDA-BIFURCACIONES DE HOPF\3_EXPONENTES DE LYAPUNOV\REACCIÓN BZ"
os.makedirs(carpeta, exist_ok=True)

archivo_excel = os.path.join(
    carpeta,
    f"lyapunov_lambda1_BZ_{len(b_values)}_a={a}.xlsx"
)

df = pd.DataFrame({
    "b": b_values,
    "lambda_1": lambda_1_list
})
df.to_excel(archivo_excel, index=False)

print(f"\nArchivo Excel guardado en:\n{archivo_excel}\n")

# ======================================================
# 5. GRÁFICA DEL EXPONENTE DE LYAPUNOV
# ======================================================

plt.figure(figsize=(10, 3))
plt.plot(b_values, lambda_1_list, lw=1.2, color="g", label=r"$\lambda_1$")

# Configurar ejes
plt.xlim(2, 5)
plt.xticks(np.arange(2, 5.1, 0.5))
plt.margins(x=0.01)

# Etiquetas
plt.xlabel(r"$b$", fontsize=13)
plt.ylabel(r"$\lambda_1$", fontsize=13)

# Línea vertical en b = 3.5 (punto de bifurcación)
plt.axvline(3.5, linestyle='--', lw=1, alpha=0.9, c="black", label=r"$b_c = 3.5$")

plt.grid(alpha=0.2)
plt.legend(loc='upper right')
plt.tight_layout()

# Guardar gráfica como PDF
plt.savefig(f"figure_bz_lyapunov_lambda1_{len(b_values)}_a={a}.pdf",
            bbox_inches="tight", dpi=600)
plt.show()